# 🏈 Fantasy Autopilot para Yahoo

Tu propio **Fantasy Plus + FantasyPros Auto-Pilot**, corriendo en Google Colab:

| Qué hace | Cómo decide |
|---|---|
| **Alineación óptima** cada semana | Suma proyecciones de Yahoo con la asignación exacta a tus posiciones (incluye el FLEX). Solo cambia algo si sube tu valor esperado. |
| **Inactivos del día de partido** | Revisa cada 10–15 min antes de cada kickoff; si un titular sale *Out*, lo cambia por el mejor suplente que todavía no haya jugado. |
| **Jugadores dudosos (Q/D)** | Calcula la probabilidad de que jueguen (estado + prácticas). Si juega de noche y no tienes un suplente que juegue igual o más tarde, cuenta el riesgo de quedarte en cero (el caso Nacua). |
| **Waivers y pujas FAAB** | Mide cuántos puntos de *alineación* te da cada agente libre en lo que queda de temporada, a quién soltar, y cuánto pujar (con tu saldo y el de tus rivales). |
| **Rival y probabilidad de ganar** | Detecta errores en la alineación del rival (titulares en IR, banca mejor que titulares). |
| **Evaluador de trades** | Totales, titulares que se van, si lo que recibes mejora a tu mejor jugador de esa posición, y puntos de alineación antes/después. |
| **Avisos al celular** | App gratis **ntfy** (sin cuenta), y opcional Telegram o correo. |

**Modos** (celda 2):
* `sugerir` — solo te avisa; no toca Yahoo. **Empieza aquí.**
* `simulacro` — prueba los cambios en Yahoo sin guardarlos (para verificar que todo funciona).
* `automatico` — guarda los cambios de alineación en Yahoo. Las pujas solo se envían si además marcas `PUJAS_AUTOMATICAS`.

**Lo que debes saber antes:**
1. Desde el 22 de julio de 2026 Yahoo bloquea su API a programas no aprobados y **no da permiso de escritura a nadie**. Por eso este notebook usa **tu sesión del navegador (cookies)**: lee las mismas páginas que ves tú y guarda con los mismos formularios.
2. Automatizar tu cuenta puede ir contra los términos de uso de Yahoo. Es tu decisión; el programa hace pocas visitas y con pausas, como una persona.
3. Las páginas de Yahoo que se leen fueron verificadas por otros proyectos en agosto–septiembre de 2026, pero **el guardado de la alineación y el formulario de pujas no se pudieron probar contra Yahoo real** desde donde se construyó esto. Por eso: primero `simulacro`, mira las capturas (celda 9), y después `automatico`.
4. Colab solo corre **mientras esta pestaña esté abierta**. Para que funcione 24/7 con la computadora apagada, usa GitHub Actions (celda 10).

In [ ]:
#@title 1. Instalar (≈1–2 min) { display-mode: "form" }
RAMA = "claude/pensive-brown-ow1zrl"  #@param {type:"string"}
INSTALAR_NAVEGADOR = True  #@param {type:"boolean"}
#@markdown El navegador (Chromium) solo hace falta para pujas automáticas, capturas y como respaldo al guardar la alineación.

!pip -q install --upgrade "git+https://github.com/delioguzmang-maker/fanatasy.git@{RAMA}"
if INSTALAR_NAVEGADOR:
    !pip -q install playwright
    !python -m playwright install --with-deps chromium > /dev/null 2>&1
print("✅ Instalado")

## 2. Tu liga

Abre tu equipo en la web de Yahoo. La dirección se ve así:

`https://football.fantasysports.yahoo.com/f1/`**`123456`**`/`**`7`**

El primer número es tu **LIGA_ID** y el segundo tu **EQUIPO_ID**.

* **NUNCA_SOLTAR**: nombres separados por coma (por ejemplo, jugadores en una oferta de trade pendiente: `Alec Pierce`).
* **PROBABILIDAD_MANUAL**: si una noticia dice algo que el estado Q/D no refleja, escribe `Puka Nacua=30%, Nico Collins=10%`.
* **TEMA_NTFY**: déjalo vacío y se crea uno secreto. Instala la app **ntfy** (Android / iPhone), toca **+** y suscríbete a ese tema.

In [ ]:
#@title 2. Configuración { display-mode: "form" }
LIGA_ID = ""  #@param {type:"string"}
EQUIPO_ID = ""  #@param {type:"string"}
MODO = "sugerir"  #@param ["sugerir", "simulacro", "automatico"]
PUJAS_AUTOMATICAS = False  #@param {type:"boolean"}
NUNCA_SOLTAR = ""  #@param {type:"string"}
PROBABILIDAD_MANUAL = ""  #@param {type:"string"}
PUJA_MAXIMA_PCT = 50  #@param {type:"slider", min:5, max:100, step:5}
TEMA_NTFY = ""  #@param {type:"string"}
GUARDAR_EN_DRIVE = True  #@param {type:"boolean"}
#@markdown `GUARDAR_EN_DRIVE` guarda capturas, historial y tu tema de ntfy en tu Google Drive (carpeta fantasy_autopilot).

import os, secrets
from fantasy_autopilot import Config

carpeta = "/content/fantasy_autopilot"
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    carpeta = "/content/drive/MyDrive/fantasy_autopilot"
os.makedirs(carpeta, exist_ok=True)

if not TEMA_NTFY:
    archivo_tema = os.path.join(carpeta, "tema_ntfy.txt")
    if os.path.exists(archivo_tema):
        TEMA_NTFY = open(archivo_tema).read().strip()
    else:
        TEMA_NTFY = "fantasy-" + secrets.token_hex(6)
        open(archivo_tema, "w").write(TEMA_NTFY)

probabilidades = {}
for parte in PROBABILIDAD_MANUAL.split(","):
    if "=" in parte:
        nombre, valor = parte.split("=", 1)
        v = float(valor.strip().rstrip("%"))
        probabilidades[nombre.strip()] = v / 100 if v > 1 else v

cfg = Config(
    league_id=LIGA_ID, team_id=EQUIPO_ID, mode=MODO, auto_claims=PUJAS_AUTOMATICAS,
    never_drop=[x.strip() for x in NUNCA_SOLTAR.split(",") if x.strip()],
    p_play_overrides=probabilidades, max_bid_pct=PUJA_MAXIMA_PCT / 100,
    ntfy_topic=TEMA_NTFY, state_dir=carpeta, quiet=True,
)
print(f"Modo: {cfg.mode} · pujas automáticas: {'sí' if cfg.auto_claims else 'no'}")
print(f"📱 En la app ntfy suscríbete al tema:  {TEMA_NTFY}")

## 3. Conectar con Yahoo (tus cookies)

Se hace **una vez** (y otra vez si Yahoo cierra tu sesión; el programa te avisa).

En una **computadora** con Chrome:
1. Entra a [football.fantasysports.yahoo.com](https://football.fantasysports.yahoo.com) con tu cuenta (marca *Mantener sesión iniciada*) y abre tu equipo.
2. Presiona **F12** → pestaña **Network** (Red) → recarga la página con **F5**.
3. Haz clic en la **primera fila** de la lista (lleva el número de tu liga).
4. En **Headers → Request Headers** busca **cookie:** y copia **todo** el valor (es largo).
5. Aquí en Colab: ícono de **llave 🔑** (Secrets) en la barra izquierda → **Add new secret** → Name: `YAHOO_COOKIES`, Value: pega → activa **Notebook access**.

⚠️ Ese valor es como tu contraseña de Yahoo: no lo pegues en chats ni lo subas a GitHub. Si cierras sesión en Yahoo deja de servir.

In [ ]:
#@title 3. Conectar y ver tu roster { display-mode: "form" }
import pandas as pd
from fantasy_autopilot import Autopilot
from fantasy_autopilot.report import when
from fantasy_autopilot.yahoo.cookies import describe, parse_cookies

try:
    from google.colab import userdata
    COOKIES = userdata.get("YAHOO_COOKIES")
except Exception:
    COOKIES = None
if not COOKIES:
    from getpass import getpass
    COOKIES = getpass("No encontré el secreto YAHOO_COOKIES. Pega aquí el valor de 'cookie' (no se muestra): ")

print("Cookies:", describe(parse_cookies(COOKIES)))
pilot = Autopilot(cfg, cookies=COOKIES)
semana = pilot.week()
roster = pilot.load_roster(semana)
pilot.enrich(roster.players, semana)
print(f"Semana {semana} · posiciones: {', '.join(roster.slots)} · banca: {roster.bench_size} · IR: {roster.ir_size}")
display(pd.DataFrame([{
    "Puesto": p.slot, "Jugador": p.name, "Pos": p.position, "Equipo": p.team,
    "Proyección": round(p.proj, 2), "Estado": p.status or "", "Juega %": round(p.prob * 100),
    "Partido": when(p.kickoff, pilot.tz), "Bloqueado": "🔒" if p.locked else "",
} for p in roster.players]))
for aviso in pilot.warnings:
    print("⚠", aviso)

In [ ]:
#@title 4. Alineación de la semana { display-mode: "form" }
#@markdown En modo `sugerir` solo te dice qué cambiar; en `automatico` lo guarda en Yahoo y lo verifica.
r = pilot.run("lineup")
print(r.title, "\n")
print(r.text)

In [ ]:
#@title 5. Tu rival y probabilidad de ganar { display-mode: "form" }
r = pilot.run("matchup")
print(r.title, "\n")
print(r.text)

In [ ]:
#@title 6. Waivers y pujas FAAB { display-mode: "form" }
#@markdown Lee ~50–80 páginas de Yahoo con pausas: tarda 1–2 minutos. Córrelo el martes antes de que se procesen las pujas.
r = pilot.run("waivers")
print(r.title, "\n")
print(r.text)
cands = r.data.get("candidates", []) if r.data else []
if cands:
    display(pd.DataFrame([{
        "Jugador": c.name, "Pos": c.position, "Equipo": c.team, "Disponible": c.owner,
        "Esta semana": round(c.weekly.get(semana, 0.0), 2),
        "Promedio resto": round(sum(c.weekly.values()) / max(1, len(c.weekly)), 2),
    } for c in cands]).sort_values("Promedio resto", ascending=False))

In [ ]:
#@title 7. Evaluar un trade { display-mode: "form" }
DOY = ""  #@param {type:"string"}
RECIBO = ""  #@param {type:"string"}
#@markdown Nombres separados por coma, por ejemplo DOY = `Nico Collins`, RECIBO = `Josh Allen`.
r = pilot.run("trade", give=[x.strip() for x in DOY.split(",") if x.strip()],
              get=[x.strip() for x in RECIBO.split(",") if x.strip()])
print(r.title, "\n")
print(r.text)

In [ ]:
#@title 8. Piloto automático del domingo (deja esta pestaña abierta) { display-mode: "form" }
HORAS = 10  #@param {type:"number"}
CADA_MINUTOS = 10  #@param {type:"integer"}
#@markdown Empieza el domingo antes de las 11:30 am (hora del Este). Solo revisa de verdad cuando falta poco para un partido;
#@markdown si un titular sale inactivo, lo cambia (modo `automatico`) o te manda una alerta urgente (modo `sugerir`).
pilot.loop(hours=HORAS, every_minutes=CADA_MINUTOS)

## 9. Calibrar la escritura en Yahoo (una vez, antes de `automatico`)

Esta celda **no guarda ni puja nada**: abre Yahoo en un navegador invisible con tu sesión, hace los clics hasta el último paso y toma **capturas**.
Mira las imágenes: en la de la alineación deben verse los menús con las posiciones nuevas; en la de la puja, marcado el jugador a soltar y escrita la cantidad.
Si algo no coincide, no uses `automatico` para eso y avísame con la captura.

In [ ]:
#@title 9. Calibrar (simulacro con capturas) { display-mode: "form" }
PROBAR_ALINEACION = True  #@param {type:"boolean"}
PROBAR_PUJA = False  #@param {type:"boolean"}
JUGADOR_A_PEDIR = ""  #@param {type:"string"}
SOLTAR = ""  #@param {type:"string"}
PUJA = 1  #@param {type:"integer"}
from IPython.display import Image

nav = pilot.browser()
if nav is None:
    raise SystemExit("Instala el navegador: vuelve a correr la celda 1 con INSTALAR_NAVEGADOR marcado.")

def mostrar(res):
    print(("✅ " if res.ok else "❌ ") + res.detail)
    for ruta in res.evidence:
        if ruta.endswith(".png"):
            display(Image(filename=ruta, width=900))

if PROBAR_ALINEACION:
    from fantasy_autopilot.optimizer import optimize_lineup, plan_moves
    plan = optimize_lineup(roster.players, roster.slots, pilot.opt_settings())
    objetivo = {m.player.pid: m.to_slot for m in plan_moves(roster.players, plan)}
    if not objetivo:  # lineup already optimal: rehearse swapping a bench player with a starter
        for b in (p for p in roster.players if p.slot == "BN" and not p.locked):
            s = next((p for p in roster.players if p.is_starter and not p.locked and b.can_fill(p.slot)), None)
            if s:
                objetivo = {s.pid: "BN", b.pid: s.slot}
                break
    print("Cambios que se ensayan (no se guardan):", objetivo)
    mostrar(nav.set_lineup(objetivo, semana, dry_run=True))

if PROBAR_PUJA:
    info = pilot.sleeper.match(name=JUGADOR_A_PEDIR) if pilot.sleeper else None
    pid = str((info or {}).get("yahoo_id") or "")
    soltar = roster.find(SOLTAR) if SOLTAR else None
    if not pid:
        print("No encontré el id de Yahoo de", JUGADOR_A_PEDIR)
    else:
        mostrar(nav.claim(pid, soltar.pid if soltar else None, PUJA, dry_run=True))

## 10. 24/7 sin Colab (GitHub Actions, gratis)

Colab se apaga cuando cierras la pestaña. El mismo programa puede correr en los servidores de GitHub con horario fijo (tu repo ya trae el archivo `.github/workflows/autopilot.yml`):

| Cuándo (hora del Este) | Qué hace |
|---|---|
| Todos los días ~10:07 am | Alineación de la semana |
| Martes ~6:37 pm | Waivers y pujas FAAB |
| Jueves, sábado, domingo y lunes, cada 15 min cerca de los partidos | Inactivos del día de partido |

Pasos (en github.com/delioguzmang-maker/fanatasy → **Settings → Secrets and variables → Actions**):
1. Pestaña **Secrets** → *New repository secret*: `YAHOO_COOKIES` (el mismo valor de la celda 3) y `NTFY_TOPIC` (tu tema).
2. Pestaña **Variables** → *New repository variable*: `FA_LEAGUE_ID`, `FA_TEAM_ID`, `FA_MODE` (`sugerir` o `automatico`) y `AUTOPILOT_ENABLED` = `true`.
3. Opcional: `FA_AUTO_CLAIMS` = `true` para que también puje (solo después de calibrar).
4. Para otras opciones (nunca soltar, probabilidades manuales) crea el secreto `FA_CONFIG_JSON`, por ejemplo:
   `{"never_drop": ["Alec Pierce"], "p_play_overrides": {"Puka Nacua": 0.3}}`

Tu repositorio es **público**: los registros de GitHub solo muestran el título de cada corrida, nunca tus cookies. GitHub solo corre horarios desde la rama principal del repo.

## Problemas frecuentes

* **"Sesión de Yahoo vencida"**: repite la celda 3 (copia de nuevo la cookie) y actualiza el secreto `YAHOO_COOKIES` (también en GitHub si lo usas).
* **"No se leyó ningún jugador"**: Yahoo cambió la página. Corre la celda 9 y comparte la captura.
* **El guardado falla en `automatico`**: el programa te manda el aviso con los cambios exactos para hacerlos a mano en la app de Yahoo (30 segundos).
* **Las pujas**: el monto es una regla, no un dato. `PUJA_MAXIMA_PCT` limita cuánto de tu saldo se puede ir en un jugador.